In [1]:
import os
from pathlib import Path
from typing import List

from tqdm import tqdm
import numpy as np
import pandas as pd
try:
    import google.generativeai as genai
except ImportError as exc:
    raise ImportError("Install google-generativeai via `pip install google-generativeai`.") from exc


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---- Configuration ----
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if GOOGLE_API_KEY is not None:
    GEMINI_API_KEY = GOOGLE_API_KEY
if not GEMINI_API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY in your environment before running this cell.")

genai.configure(api_key=GEMINI_API_KEY)

ROOT_DIR = Path("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2").expanduser()
if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

MODEL_NAME = "text-embedding-004"
BATCH_SIZE = 250  # number of .py files to embed before logging progress


In [3]:
def read_python_file(file_path: Path) -> str:
    """Return the contents of a Python file with a lightweight header."""
    return f"# File: {file_path.name}\n" + file_path.read_text(encoding="utf-8", errors="ignore")


def embed_text(texts: List[str]) -> List[float]:
    response = genai.embed_content(
        model=MODEL_NAME,
        content=texts,
        task_type="SEMANTIC_SIMILARITY",
    )
    return response["embedding"]


def embed_python_file(file_path: Path) -> np.ndarray:
    source = read_python_file(file_path)
    if not source.strip():
        raise ValueError(f"{file_path} is empty or unreadable")
    return np.asarray(embed_text(source), dtype=np.float32)


In [ ]:
records = []
pattern_dirs = [p for p in sorted(ROOT_DIR.iterdir()) if p.is_dir()]
if not pattern_dirs:
    raise ValueError(f"No pattern directories detected in {ROOT_DIR}")

python_files = []
for pattern_dir in pattern_dirs:
    python_files.extend((pattern_dir.name, py_file) for py_file in sorted(pattern_dir.glob("**/*.py")))

if not python_files:
    raise ValueError("No .py files found under the supplied root directory.")

total_files = len(python_files)
for batch_start in range(0, total_files, BATCH_SIZE):
    batch = python_files[batch_start : batch_start + BATCH_SIZE]
    print(
        f"Processing files {batch_start + 1}-{batch_start + len(batch)} / {total_files}"
    )

    texts = [read_python_file(py_path) for _, py_path in batch]
    embeddings = embed_text(texts)
    for (pattern_name, py_path), embedding_vector in zip(batch, embeddings):
        records.append(
            {
                "pattern": pattern_name,
                "file": str(py_path.relative_to(ROOT_DIR)),
                "embedding": embedding_vector,
            }
        )
    
    # for pattern_name, py_path in tqdm(batch, desc="Files", leave=False):
    #     try:
    #         embedding_vector = embed_python_file(py_path)
    #     except ValueError as err:
    #         print(f"Skipping {py_path}: {err}")
    #         continue
    #     records.append(
    #         {
    #             "pattern": pattern_name,
    #             "file": str(py_path.relative_to(ROOT_DIR)),
    #             "embedding": embedding_vector,
    #         }
    #     )

if not records:
    raise ValueError("No embeddings were generated; ensure .py files have content.")

embedding_length = len(records[0]["embedding"])
rows = []
for record in records:
    row = {f"dim_{i+1}": value for i, value in enumerate(record["embedding"])}
    row["pattern"] = record["pattern"]
    row["file"] = record["file"]
    rows.append(row)

embeddings_df = pd.DataFrame(rows)
print(
    f"Generated embeddings for {len(embeddings_df)} files across {len(pattern_dirs)} patterns with {embedding_length} dimensions."
)
display(embeddings_df.head())

OUTPUT_PATH = Path("./results/pattern_embeddings/gemini_pattern_embedding_v3.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
embeddings_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved embeddings to {OUTPUT_PATH.resolve()}")


Processing files 1-250 / 2177


NotFound: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.

In [ ]:
embeddings_df.shape